In [1]:
import socket
import threading
import time

In [2]:
host = "127.0.0.1"
port = 65005
max_worker = 2

In [3]:
worker_slot = threading.Semaphore(max_worker)

In [4]:
def handle_client(conn, addr):
    with conn:
        request = conn.recv(1024).decode();
        print(f"[worker-{threading.get_ident()}] waiting for free slot")

        got_slot = worker_slot.acquire(timeout=3)

        if not got_slot:
            print(f"[worker-{threading.get_ident()} no slot free")
            conn.sendall("Server busy: please try again later!".encode())
            return
        try:
            print(f"[worker-{threading.get_ident()} got a slot, working on {addr}")
            time.sleep(2)
            reply = f"Processed '{request}' by worker thread {threading.get_ident()}"
            conn.sendall(reply.encode())
            print(f"[worker-{threading.get_ident()}] Done with {addr}, releasing slot")
        finally:
            worker_slot.release()

In [5]:
def main():
    with socket.socket(socket.AF_INET,socket.SOCK_STREAM) as s:
        s.setsockopt(socket.SOL_SOCKET,socket.SO_REUSEADDR,1)
        s.bind((host,port))
        s.listen(5)

        print(f"[dispather] listening on {host}:{port} (max {max_worker} concurrent workers")

        while True:
            conn,addr = s.accept()
            print(f"[dispather] accepted {addr} , spawing worker thread")
            worker = threading.Thread(
                target=handle_client,
                args=(conn,addr)
            )
            worker.start()

In [6]:
if __name__ == "__main__":
    main()

[dispather] listening on 127.0.0.1:65005 (max 2 concurrent workers
[dispather] accepted ('127.0.0.1', 57986) , spawing worker thread
[worker-123863367153344] waiting for free slot
[worker-123863367153344 got a slot, working on ('127.0.0.1', 57986)
[dispather] accepted ('127.0.0.1', 57988) , spawing worker thread
[worker-123863358760640] waiting for free slot
[worker-123863358760640 got a slot, working on ('127.0.0.1', 57988)
[dispather] accepted ('127.0.0.1', 57992) , spawing worker thread
[worker-123863350367936] waiting for free slot
[dispather] accepted ('127.0.0.1', 58008) , spawing worker thread
[worker-123863341975232] waiting for free slot
[dispather] accepted ('127.0.0.1', 58020) , spawing worker thread
[worker-123863333582528] waiting for free slot
[worker-123863358760640] Done with ('127.0.0.1', 57988), releasing slot
[worker-123863350367936 got a slot, working on ('127.0.0.1', 57992)
[worker-123863367153344] Done with ('127.0.0.1', 57986), releasing slot
[worker-123863341975

KeyboardInterrupt: 